## LangGraph Agent with tool calling

<p>
    Below is an end to end demonstration of a LangGraph agent. 
    The agent is using Ollama's Llama3.2 locally.
</p>

**Demonstrates:**
- Tool binding
- State management with MessagesState
- Conditional routing
- End to end agent execution

## Setup
Install and import required libraries.

In [5]:
!pip install langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Uncomment to verify LLM is working
# response = llm.invoke("What is 2+2?")
# print(response.content)

2 + 2 = 4.


In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers together. Use this when you need to multiply."""
    return a * b

# Testing the tool directly
print(multiply.invoke({"a": 3, "b": 4}))

12.0


In [5]:
from langchain_core.messages import HumanMessage
# Give the tool to the LLM
llm_with_tools = llm.bind_tools([multiply])

In [6]:
# Ask it something that requires multiplication
response = llm_with_tools.invoke("What is 6 multiplied by 7?")
print(response.tool_calls)

[{'name': 'multiply', 'args': {'a': 6, 'b': 7}, 'id': 'c468078d-cbd1-4f2a-b08d-4a90ee9fc748', 'type': 'tool_call'}]


#### When the LLM receives a question, it doesn't always answer directly. Instead it can decide to use a tool. Here we extract that decision from the response, and manually execute the tool it chose; in this case multiply. The tool name lives in 'name' and the inputs live in 'args'.

In [7]:
# Get the tool call from the response
tool_call = response.tool_calls[0]

# Execute it manually
tool_result = multiply.invoke(tool_call["args"])
print(f"Tool used: {tool_call['name']}")
print(f"Result: {tool_result}")

Tool used: multiply
Result: 42.0


In [8]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# Define the graph with built-in MessagesState
graph_builder = StateGraph(MessagesState)

In [9]:
# Node 1 — the LLM that decides what to do
def call_llm(state: MessagesState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Node 2 — executes whatever tool the LLM chose
tool_node = ToolNode([multiply])

# Add both nodes to the graph
graph_builder.add_node("llm", call_llm)
graph_builder.add_node("tools", tool_node)

In [10]:
from langgraph.prebuilt import tools_condition

# Where to start
graph_builder.add_edge(START, "llm")

# After LLM — go to tools if needed, otherwise end
graph_builder.add_conditional_edges("llm", tools_condition)

# After tools — always go back to LLM
graph_builder.add_edge("tools", "llm")

# Compile the graph
graph = graph_builder.compile()

In [11]:
# Run the agent!
result = graph.invoke({
    "messages": [HumanMessage(content="What is 6 multiplied by 7?")]
})

# Print the final answer
print(result["messages"][-1].content)

The answer to the question "What is 6 multiplied by 7?" is 42.
